<a href="https://colab.research.google.com/github/sumoham20/ITCS-3162/blob/main/lab_02_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Example: AI Tools & the Data Mining Pipeline

**ITCS 3162 — Introduction to Data Mining**

This lab has two goals:
1. Set expectations for how to use AI assistants (ChatGPT, Claude, Copilot, Gemini, etc.) **responsibly and effectively** in this course.
2. Introduce the **end-to-end pipeline** every data mining project follows — so you have a mental map for every lab that comes after this one.

## Learning objectives
- Explain the difference between *appropriate* and *inappropriate* AI use in coursework
- Write a clear, well-scoped prompt and critically evaluate the response
- Describe the stages of a typical data mining pipeline
- Recognize how a notebook-based workflow maps onto a cloud/production pipeline


## Part 1 — Using AI tools responsibly

AI assistants are useful, but they are tools — like a calculator, a textbook, or a tutor. You are responsible for everything you submit. The goal of this course is for **you** to learn data mining, not for an AI to do the work for you.

### Course AI policy (summary)

**Appropriate use ✅**
- Explaining a concept you don't understand ("What does stratified sampling mean?")
- Debugging an error message you've already tried to understand
- Suggesting library functions when you know what you want to accomplish
- Reviewing your code for style or clarity *after* you've written it
- Generating practice questions to study

**Inappropriate use ❌**
- Pasting an entire exercise prompt and submitting the AI's answer
- Asking the AI to write reflection / interpretation answers for you
- Using AI on quizzes or exams when not explicitly allowed
- Using AI output without verifying it works and understanding why

> A good test: **could you defend every line of your submission in a 2-minute conversation with the instructor?** If not, you're using AI as a substitute for learning, not a support.

### Why this matters technically

LLMs are confident even when wrong — this is called **hallucination**. They will invent functions that don't exist, cite papers that were never written, and confidently misremember API arguments. In data mining specifically, they often:
- Use deprecated sklearn syntax
- Mix up `fit`, `transform`, and `fit_transform`
- Suggest leaky preprocessing (fitting a scaler on the full dataset before train/test split)

You must run, test, and read everything you accept.


### Prompt quality: a worked example

Compare these two prompts for the same goal.

**Weak prompt:**
> "write code to do regression"

**Strong prompt:**
> "I have a pandas DataFrame `df` with numeric columns `age`, `bmi`, `bp` and a target `charges`. Write scikit-learn code to: (1) split into 80/20 train/test, (2) fit a linear regression, (3) report R² and RMSE on the test set. Use `random_state=42`."

The strong prompt names the library, the data shape, the target, the steps, the metrics, and reproducibility — so the answer is specific and verifiable. Below is roughly what a good AI assistant should produce for it. Read the code carefully!


In [ ]:
# Realistic example of what an AI assistant would produce for the strong prompt above.
# We use a small synthetic DataFrame so this cell is self-contained.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    "age": rng.integers(18, 65, n),
    "bmi": rng.normal(27, 4, n),
    "bp":  rng.normal(120, 15, n),
})
# A made-up linear relationship + noise so the model has something to learn
df["charges"] = 200*df["age"] + 350*df["bmi"] + 50*df["bp"] + rng.normal(0, 1500, n)

X = df[["age", "bmi", "bp"]]
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"R² : {r2_score(y_test, y_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):,.2f}")

R² : 0.849
RMSE: 1,420.66


### Verifying AI output — a checklist

For every chunk of AI-generated code, ask:
1. **Does it run?** (Did you actually execute it?)
2. **Do I understand each line?** (Could you explain what `train_test_split` returns?)
3. **Did it use the right tool?** (Linear regression is fine for continuous targets; would fail for classification.)
4. **Is there a data leak?** (Were preprocessing steps fit only on training data?)
5. **Does the result make sense?** (R² of 0.999 on real-world data is a red flag, not a win.)


## Part 2 — The data mining pipeline

Almost every project in this course (and in industry) follows the same stages. Different textbooks name them differently — *CRISP-DM*, *KDD*, *the ML lifecycle* — but the idea is the same.


### The six stages

| # | Stage | What happens | Course lab |
|---|---|---|---|
| 1 | **Business / problem understanding** | What question are we answering? Who's the user? What's "success"? | every lab |
| 2 | **Data acquisition & loading** | Get the data into a usable structure (DataFrame, etc.) | Lab 3 |
| 3 | **Exploration & visualization** | Summary stats, distributions, missingness, outliers | Lab 4 |
| 4 | **Preprocessing & feature engineering** | Cleaning, encoding, scaling, dimensionality reduction | Lab 5 |
| 5 | **Modeling** | Choose algorithm, train, tune | Labs 7–10 |
| 6 | **Evaluation & deployment** | Metrics, validation, putting the model to use | Lab 6 framing |

The arrows aren't strictly one-way — you'll loop back constantly. Discovering bad data in stage 5 sends you back to stage 4.


In [ ]:
# A miniature pipeline: load → explore → preprocess → model → evaluate
# This is the same five-stage shape you'll see throughout the course,
# compressed into one cell.

import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. LOAD
iris = load_iris(as_frame=True)
df = iris.frame
print("Stage 1-2 (load):", df.shape, "rows x cols")

# 2. EXPLORE (one line — we'll go deeper in Lab 4)
print("\nStage 3 (explore) — class balance:")
print(df["target"].value_counts().to_string())

# 3. PREPROCESS — split first, then scale (avoid leakage)
X = df.drop(columns="target")
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)
print("\nStage 4 (preprocess): scaler fit on train only")

# 4. MODEL
model = LogisticRegression(max_iter=200).fit(X_train_s, y_train)
print("Stage 5 (model): trained logistic regression")

# 5. EVALUATE
acc = accuracy_score(y_test, model.predict(X_test_s))
print(f"\nStage 6 (evaluate): test accuracy = {acc:.3f}")

Stage 1-2 (load): (150, 5) rows x cols

Stage 3 (explore) — class balance:
target
0    50
1    50
2    50

Stage 4 (preprocess): scaler fit on train only
Stage 5 (model): trained logistic regression

Stage 6 (evaluate): test accuracy = 0.921


Look at the structure of that cell — every lab notebook from here forward will follow some subset of those five stages.


## Part 3 — From notebook to cloud pipeline

We'll work in notebooks all semester because they're great for **exploration**: small data, fast iteration, lots of inline visualization. But in production, the same pipeline looks different.

### sklearn `Pipeline` — the bridge

Even in a notebook, you can express your stages as a single composable object. This is the conceptual building block for production deployments.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Exactly the same modeling as the cell above, but bundled
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=200)),
])

pipe.fit(X_train, y_train)              # scaler.fit_transform → model.fit
acc = pipe.score(X_test, y_test)        # scaler.transform → model.score
print(f"Pipeline test accuracy: {acc:.3f}")

# Bonus: pipe can be pickled and shipped to a server as a single object.

Pipeline test accuracy: 0.921


### Why this matters in the cloud

The same logical pipeline shows up in tools like **AWS SageMaker**, **Azure ML**, **Google Vertex AI**, and **Databricks**, just with each stage broken out:

```
┌────────────┐   ┌────────────┐   ┌────────────┐   ┌────────────┐   ┌────────────┐
│  Ingest    │ → │  Process   │ → │  Train     │ → │  Evaluate  │ → │  Deploy    │
│ (S3, DB)   │   │ (Spark,    │   │ (managed   │   │ (metrics,  │   │ (REST API, │
│            │   │  pandas)   │   │  training) │   │  dashboards│   │  batch job)│
└────────────┘   └────────────┘   └────────────┘   └────────────┘   └────────────┘
                                        ↑                                  │
                                        └────────── monitoring & retraining ┘
```

Key differences from a notebook:
- Each stage is its **own service**, not a cell — they can run on different schedules and different machines.
- Data is **versioned** (DVC, Delta Lake, lakeFS) and so are **models** (MLflow, SageMaker Model Registry).
- The pipeline is **scheduled**, not run by hand (Airflow, Step Functions, Vertex Pipelines).
- The notebook is for **experimentation**; the deployed pipeline is for **serving predictions** on new data, often at scale.

You don't need to know how to deploy any of this for the course — but every time you write `pipe.fit(...)` in a notebook, you're prototyping the middle of that diagram.


## Summary

- AI tools are useful when used as a tutor and a debugger; risky when used as a shortcut.
- Strong prompts include the data, library, steps, and metrics.
- Always verify AI output (does it run? do you understand it? is there leakage?).
- The data mining pipeline has six stages: problem → load → explore → preprocess → model → evaluate.
- `sklearn.Pipeline` lets you express stages as one composable object — and the same shape scales up to cloud platforms.

Open **`lab_02_exercise.ipynb`** to practice.
